# Trial evaluation: QTDB noise type 1

This Colab notebook evaluates an intermediate checkpoint from the hybrid experiment:

- Paper-A QTDB preprocessing from consecutive `pu1` P-wave onsets.
- Paper-A `noise_type=1` test corruption: second half of NSTDB `bw`, channel 2.
- Research-B 1-channel U-Net with HNF, Bridge/FiLM, and self-attention.

The default run is intentionally bounded to 20 beats per test record and 1-shot DDPM. It is a diagnostic run, not the final merged type-1/type-2 report.


## 1. Colab setup


In [ ]:
import gc
import json
import math
import subprocess
import sys
import time
from pathlib import Path

try:
    import wfdb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
    import wfdb

import numpy as np
import pandas as pd
from scipy import signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Google Drive was not mounted.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a GPU runtime before running diffusion evaluation.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Evaluation configuration


In [ ]:
CHECKPOINT_PATH = Path(
    '/content/drive/MyDrive/phase1/checkpoints/'
    'qtdb_1ch_Adata_Bmodel_multidomain_noise_type_1_last.pth'
)
OUTPUT_DIR = Path('/content/drive/MyDrive/phase1/results/qtdb_type1_trial')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_RECORDS = [
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
]

SEED = 1234
TARGET_FS = 360
BEAT_LENGTH = 512
MAX_UNPADDED_LENGTH = 496
INSERT_OFFSET = 16
P_ONSET_RETRACTION_MS = 40
REFLECT_PAD_MS = 100
MAX_BEATS_PER_RECORD = 20

# Start with [1]. Add 3/5/10 only after the 1-shot result is valid.
SHOTS = [1]
BATCH_SIZE = 8
NUM_DIFFUSION_STEPS = 50
BETA_START = 1e-4
BETA_END = 0.5
BETA_SCHEDULE = 'quad'
BASE_FEATS = 80
EMB_DIM = 128
NOISE_LEVEL_VALUES = np.arange(20, 200, dtype=np.int16) / 100.0
NOISE_BINS = [
    ('0.2-0.6', 0.2, 0.6),
    ('0.6-1.0', 0.6, 1.0),
    ('1.0-1.5', 1.0, 1.5),
    ('1.5-2.0', 1.5, 2.0),
]

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('Checkpoint:', CHECKPOINT_PATH)


## 3. Research-B architecture


In [ ]:
class HNFBlockUNet(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k, padding=k // 2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        out = torch.cat([conv(x) for conv in self.multi_convs], dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out = torch.cat([self.half_inst_norm(out[:, :half, :]), out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)


class BridgeBlockUNet(nn.Module):
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(nn.Linear(emb_dim, features * 2), nn.SiLU())

    def sinusoidal_embedding(self, x):
        x = x.view(-1)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)

    def forward(self, x, alpha_bar):
        emb = self.sinusoidal_embedding(alpha_bar)
        scale, shift = self.film(emb).chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(-1)) + shift.unsqueeze(-1)


class SelfAttention1D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, length = x.shape
        qkv = self.qkv(x).reshape(batch, 3, self.num_heads, self.head_dim, length)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        return self.proj(out.reshape(batch, channels, length))


class UNet1D(nn.Module):
    def __init__(self, in_channels=2, base_channels=80, emb_dim=128, out_channels=1):
        super().__init__()
        self.enc1 = HNFBlockUNet(in_channels, base_channels)
        self.bridge1 = BridgeBlockUNet(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = HNFBlockUNet(base_channels * 2, base_channels * 2)
        self.bridge2 = BridgeBlockUNet(base_channels * 2, emb_dim)
        self.down2 = nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = HNFBlockUNet(base_channels * 4, base_channels * 4)
        self.bridge3 = BridgeBlockUNet(base_channels * 4, emb_dim)
        self.down3 = nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=4, stride=2, padding=1)
        self.enc4 = HNFBlockUNet(base_channels * 8, base_channels * 8)
        self.bridge4 = BridgeBlockUNet(base_channels * 8, emb_dim)
        self.attn = SelfAttention1D(base_channels * 8)
        self.up4 = nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlockUNet(base_channels * 8, base_channels * 4)
        self.up3 = nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlockUNet(base_channels * 4, base_channels * 2)
        self.up2 = nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlockUNet(base_channels * 2, base_channels)
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        inp = torch.cat([x, cond], dim=1)
        e1 = self.bridge1(self.enc1(inp), noise_scale)
        e2 = self.bridge2(self.enc2(self.down1(e1)), noise_scale)
        e3 = self.bridge3(self.enc3(self.down2(e2)), noise_scale)
        e4 = self.bridge4(self.enc4(self.down3(e3)), noise_scale)
        e4 = self.attn(e4)
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        return self.final(d2)


## 4. Matching 50-step conditional DDPM


In [ ]:
def make_beta_schedule(schedule_name, num_steps, start, end):
    if schedule_name == 'linear':
        return torch.linspace(start, end, num_steps)
    if schedule_name == 'quad':
        return torch.linspace(start ** 0.5, end ** 0.5, num_steps) ** 2
    if schedule_name == 'sigmoid':
        values = torch.linspace(-6, 6, num_steps)
        return torch.sigmoid(values) * (end - start) + start
    raise ValueError(schedule_name)


class DDPM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.num_steps = NUM_DIFFUSION_STEPS
        betas = make_beta_schedule(
            BETA_SCHEDULE, NUM_DIFFUSION_STEPS, BETA_START, BETA_END
        )
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = torch.cat([torch.ones(1), alphas_cumprod[:-1]])
        continuous_boundaries = torch.sqrt(torch.cat([torch.ones(1), alphas_cumprod]))
        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

        self.register_buffer('betas', betas.float())
        self.register_buffer('alphas_cumprod', alphas_cumprod.float())
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev.float())
        self.register_buffer('continuous_boundaries', continuous_boundaries.float())
        self.register_buffer('posterior_variance', posterior_variance.float())
        self.register_buffer(
            'posterior_log_variance_clipped',
            torch.log(torch.clamp(posterior_variance, min=1e-20)).float(),
        )
        self.register_buffer(
            'posterior_mean_coef1',
            (betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)).float(),
        )
        self.register_buffer(
            'posterior_mean_coef2',
            ((1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) /
             (1.0 - alphas_cumprod)).float(),
        )

    def q_posterior(self, x_start, x_t, timestep):
        coef1 = self.posterior_mean_coef1[timestep].view(-1, 1, 1)
        coef2 = self.posterior_mean_coef2[timestep].view(-1, 1, 1)
        mean = coef1 * x_start + coef2 * x_t
        log_variance = self.posterior_log_variance_clipped[timestep].view(-1, 1, 1)
        return mean, log_variance

    @torch.no_grad()
    def sample(self, condition, num_shots=1):
        output_sum = torch.zeros_like(condition)
        batch = condition.shape[0]
        for _ in range(num_shots):
            x = torch.randn_like(condition)
            for step in reversed(range(self.num_steps)):
                timestep = torch.full((batch,), step, device=x.device, dtype=torch.long)
                noise_level = self.continuous_boundaries[step + 1].expand(batch, 1)
                predicted_noise = self.model(x, condition, noise_level)
                alpha_bar = self.alphas_cumprod[step]
                x0_prediction = (
                    x - torch.sqrt(1.0 - alpha_bar) * predicted_noise
                ) / torch.sqrt(alpha_bar)
                posterior_mean, posterior_log_variance = self.q_posterior(
                    x0_prediction, x, timestep
                )
                if step > 0:
                    x = posterior_mean + torch.exp(0.5 * posterior_log_variance) * torch.randn_like(x)
                else:
                    x = posterior_mean
            output_sum += x
        return output_sum / num_shots


## 5. Load and verify the intermediate checkpoint


In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT_PATH}. Upload it to Drive or edit CHECKPOINT_PATH.'
    )

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
    checkpoint_epoch = checkpoint.get('epoch')
    checkpoint_noise_type = checkpoint.get('noise_type')
    checkpoint_metrics = checkpoint.get('metrics', {})
    checkpoint_config = checkpoint.get('config', {})
else:
    state_dict = checkpoint
    checkpoint_epoch = None
    checkpoint_noise_type = None
    checkpoint_metrics = {}
    checkpoint_config = {}

if checkpoint_noise_type not in (None, 1):
    raise RuntimeError(f'Expected noise_type=1 checkpoint, found {checkpoint_noise_type}.')
if checkpoint_config:
    expected_objective = 'research_b_l1_mean_plus_snr_weighted_relative_stft_x0'
    if checkpoint_config.get('objective') != expected_objective:
        raise RuntimeError(
            'This checkpoint does not use the stable multi-domain objective. '
            f"Found: {checkpoint_config.get('objective')}"
        )

base_model = UNet1D(
    in_channels=2, base_channels=BASE_FEATS,
    emb_dim=EMB_DIM, out_channels=1,
).to(DEVICE)
model = DDPM(base_model).to(DEVICE)
state_dict = {key.replace('module.', ''): value for key, value in state_dict.items()}
missing, unexpected = model.load_state_dict(state_dict, strict=False)
model_missing = [key for key in missing if key.startswith('model.')]
model_unexpected = [key for key in unexpected if key.startswith('model.')]
if model_missing or model_unexpected:
    raise RuntimeError(
        f'Architecture mismatch. Missing model keys: {model_missing[:5]}; '
        f'unexpected model keys: {model_unexpected[:5]}'
    )
model.eval()

print('Loaded epoch:', checkpoint_epoch)
print('Saved metrics:', checkpoint_metrics)
print('Diffusion-buffer differences:', len(missing) - len(model_missing), len(unexpected) - len(model_unexpected))


## 6. Build Paper-A test beats


In [ ]:
BEAT_SYMBOLS = {
    'N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r',
    'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?'
}


def read_annotation(record_name, extension):
    return wfdb.rdann(record_name, extension, pn_dir='qtdb/1.0.0')


def p_wave_onsets(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample, dtype=np.int64)
    onsets = []
    for index, symbol in enumerate(symbols):
        if symbol != '(':
            continue
        lookahead = symbols[index + 1:min(index + 4, len(symbols))]
        if 'p' in lookahead:
            onsets.append(int(samples[index]))
    return np.asarray(onsets, dtype=np.int64)


def beat_locations(annotation):
    return np.asarray([
        sample for sample, symbol in zip(annotation.sample, annotation.symbol)
        if symbol in BEAT_SYMBOLS
    ], dtype=np.int64)


def resample_with_reflect_padding(segment, source_fs):
    if len(segment) < 3:
        return None
    pad_source = max(1, int(round(REFLECT_PAD_MS * source_fs / 1000.0)))
    pad_source = min(pad_source, len(segment) - 1)
    padded = np.pad(segment, (pad_source, pad_source), mode='reflect')
    target_length = int(round(len(padded) * TARGET_FS / float(source_fs)))
    resampled = signal.resample(padded, target_length).astype(np.float32)
    pad_target = int(round(pad_source * TARGET_FS / float(source_fs)))
    expected_length = int(round(len(segment) * TARGET_FS / float(source_fs)))
    return resampled[pad_target:pad_target + expected_length]


def build_record_beats(record_name):
    record = wfdb.rdrecord(record_name, pn_dir='qtdb/1.0.0', channels=[0])
    pu = read_annotation(record_name, 'pu1')
    atr = read_annotation(record_name, 'atr')
    ecg = record.p_signal[:, 0].astype(np.float32)
    source_fs = float(record.fs)
    shifted = p_wave_onsets(pu) - int(round(P_ONSET_RETRACTION_MS * source_fs / 1000.0))
    shifted = shifted[(shifted >= 0) & (shifted < len(ecg))]
    r_locations = beat_locations(atr)
    beats, metadata = [], []

    for segment_index, (start, end) in enumerate(zip(shifted[:-1], shifted[1:])):
        if end <= start + 2:
            continue
        r_count = int(np.sum((r_locations >= start) & (r_locations < end)))
        if r_count > 1:
            continue
        segment = resample_with_reflect_padding(ecg[start:end], source_fs)
        if segment is None or len(segment) > MAX_UNPADDED_LENGTH:
            continue
        segment = segment - 0.5 * (float(segment[0]) + float(segment[-1]))
        clean = np.zeros(BEAT_LENGTH, dtype=np.float32)
        clean[INSERT_OFFSET:INSERT_OFFSET + len(segment)] = segment
        if np.all(np.isfinite(clean)):
            beats.append(clean[:, None])
            metadata.append({
                'record': record_name,
                'segment_index': segment_index,
                'unpadded_length': len(segment),
            })
        if MAX_BEATS_PER_RECORD is not None and len(beats) >= MAX_BEATS_PER_RECORD:
            break
    return beats, metadata


test_beats, test_metadata = [], []
for record_name in tqdm(TEST_RECORDS, desc='QTDB test records'):
    record_beats, record_metadata = build_record_beats(record_name)
    if not record_beats:
        raise RuntimeError(f'No valid Paper-A beat was produced for {record_name}.')
    test_beats.extend(record_beats)
    test_metadata.extend(record_metadata)
    print(record_name, len(record_beats))

clean_beats = np.stack(test_beats).astype(np.float32)
metadata_frame = pd.DataFrame(test_metadata)
print('Test beats:', clean_beats.shape)
display(metadata_frame.groupby('record').size().rename('beats').to_frame())


## 7. Create deterministic noise-type-1 test pairs


In [ ]:
def load_type1_test_noise():
    record = wfdb.rdrecord('bw', pn_dir='nstdb/1.0.0')
    noise = record.p_signal.astype(np.float32)
    if int(record.fs) != TARGET_FS:
        target_length = int(round(len(noise) * TARGET_FS / float(record.fs)))
        noise = signal.resample(noise, target_length, axis=0).astype(np.float32)
    if noise.shape[1] < 2:
        raise RuntimeError('NSTDB bw must have two channels.')
    midpoint = len(noise) // 2
    return noise[midpoint:, 1].copy()


def amplitude_range(values, eps=1e-8):
    return float(np.max(values) - np.min(values) + eps)


test_noise = load_type1_test_noise()
rng = np.random.default_rng(SEED + 1)
noisy_beats = np.empty_like(clean_beats)
noise_levels = rng.choice(NOISE_LEVEL_VALUES, size=len(clean_beats)).astype(np.float32)
noise_starts = rng.integers(
    0, len(test_noise) - BEAT_LENGTH, size=len(clean_beats), dtype=np.int64
)

for index, clean in enumerate(clean_beats):
    start = int(noise_starts[index])
    patch = test_noise[start:start + BEAT_LENGTH, None]
    alpha = float(noise_levels[index]) * amplitude_range(clean) / amplitude_range(patch)
    noisy_beats[index] = clean + alpha * patch

if not np.all(np.isfinite(noisy_beats)):
    raise FloatingPointError('Noisy test set contains NaN or Inf.')
print('Clean/noisy:', clean_beats.shape, noisy_beats.shape)
print('Noise-level range:', float(noise_levels.min()), float(noise_levels.max()))


## 8. Metric definitions


In [ ]:
METRIC_COLUMNS = [
    'SSD', 'MAD', 'PRD_A_code', 'PRD_standard',
    'Cosine similarity', 'SNR in', 'SNR out', 'SNR improvement',
]


def metric_frame(clean, noisy, prediction, shot, method, start_index):
    clean_flat = clean.reshape(len(clean), -1)
    noisy_flat = noisy.reshape(len(noisy), -1)
    prediction_flat = prediction.reshape(len(prediction), -1)
    error_power = np.sum((prediction_flat - clean_flat) ** 2, axis=1)
    input_error_power = np.sum((noisy_flat - clean_flat) ** 2, axis=1)
    signal_power = np.sum(clean_flat ** 2, axis=1)
    clean_mean = np.mean(clean_flat, axis=1, keepdims=True)
    prd_a_denominator = np.sum((prediction_flat - clean_mean) ** 2, axis=1)
    prd_standard_denominator = np.sum((clean_flat - clean_mean) ** 2, axis=1)
    cosine = np.sum(clean_flat * prediction_flat, axis=1) / (
        np.linalg.norm(clean_flat, axis=1) * np.linalg.norm(prediction_flat, axis=1) + 1e-8
    )
    snr_in = 10.0 * np.log10((signal_power + 1e-8) / (input_error_power + 1e-8))
    snr_out = 10.0 * np.log10((signal_power + 1e-8) / (error_power + 1e-8))
    indices = np.arange(start_index, start_index + len(clean))
    return pd.DataFrame({
        'sample_index': indices,
        'record': metadata_frame.iloc[indices]['record'].to_numpy(),
        'noise_type': 1,
        'noise_level': noise_levels[indices],
        'shot': shot,
        'method': method,
        'SSD': error_power,
        'MAD': np.max(np.abs(prediction_flat - clean_flat), axis=1),
        'PRD_A_code': 100.0 * np.sqrt(error_power / (prd_a_denominator + 1e-8)),
        'PRD_standard': 100.0 * np.sqrt(error_power / (prd_standard_denominator + 1e-8)),
        'Cosine similarity': cosine,
        'SNR in': snr_in,
        'SNR out': snr_out,
        'SNR improvement': snr_out - snr_in,
    })


## 9. Run streaming evaluation


In [ ]:
result_parts = []
sample_plot = None
evaluation_start = time.time()

for shot in SHOTS:
    for batch_start in tqdm(range(0, len(clean_beats), BATCH_SIZE), desc=f'{shot}-shot'):
        batch_end = min(batch_start + BATCH_SIZE, len(clean_beats))
        clean_batch = clean_beats[batch_start:batch_end]
        noisy_batch = noisy_beats[batch_start:batch_end]
        noisy_tensor = torch.from_numpy(noisy_batch).permute(0, 2, 1).to(DEVICE)

        with torch.no_grad():
            denoised_tensor = model.sample(noisy_tensor, num_shots=shot)
        if not torch.isfinite(denoised_tensor).all():
            raise FloatingPointError(
                f'Model output contains NaN/Inf at batch {batch_start}, shot={shot}. '
                'The checkpoint is not valid for evaluation.'
            )
        denoised_batch = denoised_tensor.cpu().permute(0, 2, 1).numpy().astype(np.float32)

        result_parts.append(metric_frame(
            clean_batch, noisy_batch, noisy_batch,
            shot, 'Noisy input', batch_start,
        ))
        result_parts.append(metric_frame(
            clean_batch, noisy_batch, denoised_batch,
            shot, 'Denoised', batch_start,
        ))

        if sample_plot is None:
            sample_plot = {
                'clean': clean_batch[0].copy(),
                'noisy': noisy_batch[0].copy(),
                'denoised': denoised_batch[0].copy(),
                'record': metadata_frame.iloc[batch_start]['record'],
                'shot': shot,
            }
        del noisy_tensor, denoised_tensor, denoised_batch
        torch.cuda.empty_cache()

metrics_per_beat = pd.concat(result_parts, ignore_index=True)
print('Elapsed minutes:', round((time.time() - evaluation_start) / 60.0, 2))
display(metrics_per_beat.head())


## 10. Build mean and standard-deviation tables


In [ ]:
def add_group_labels(frame):
    frame = frame.copy()
    frame['Group'] = 'ALL'
    grouped = [frame]
    for label, low, high in NOISE_BINS:
        if high < 2.0:
            mask = (frame['noise_level'] >= low) & (frame['noise_level'] < high)
        else:
            mask = (frame['noise_level'] >= low) & (frame['noise_level'] <= high)
        part = frame.loc[mask].copy()
        part['Group'] = label
        grouped.append(part)
    return pd.concat(grouped, ignore_index=True)


grouped_metrics = add_group_labels(metrics_per_beat)
summary_rows = []
for (shot, method, group), part in grouped_metrics.groupby(['shot', 'method', 'Group'], sort=False):
    row = {'Shot': shot, 'Method': method, 'Group': group, 'N': len(part)}
    for metric in METRIC_COLUMNS:
        row[f'{metric}_mean'] = float(part[metric].mean())
        row[f'{metric}_std'] = float(part[metric].std(ddof=1))
    summary_rows.append(row)
summary_numeric = pd.DataFrame(summary_rows)

summary_display = summary_numeric[['Shot', 'Method', 'Group', 'N']].copy()
for metric in METRIC_COLUMNS:
    summary_display[metric] = summary_numeric.apply(
        lambda row: f"{row[f'{metric}_mean']:.4f} +/- {row[f'{metric}_std']:.4f}", axis=1
    )
display(summary_display)


## 11. Diagnostic comparison


In [ ]:
all_rows = summary_numeric[summary_numeric['Group'] == 'ALL'].copy()
diagnostic = all_rows[[
    'Shot', 'Method', 'Cosine similarity_mean',
    'SNR out_mean', 'SNR improvement_mean', 'PRD_standard_mean',
]]
display(diagnostic)

first_shot = SHOTS[0]
noisy_cosine = diagnostic.loc[
    (diagnostic['Shot'] == first_shot) & (diagnostic['Method'] == 'Noisy input'),
    'Cosine similarity_mean',
].iloc[0]
denoised_cosine = diagnostic.loc[
    (diagnostic['Shot'] == first_shot) & (diagnostic['Method'] == 'Denoised'),
    'Cosine similarity_mean',
].iloc[0]
print(f'Noisy cosine: {noisy_cosine:.4f}')
print(f'Denoised cosine: {denoised_cosine:.4f}')
if denoised_cosine <= noisy_cosine:
    print('WARNING: the checkpoint reduces waveform similarity; continue training or inspect sampling.')
else:
    print('Checkpoint improves cosine over the noisy-input baseline on this trial subset.')


## 12. Plot one representative beat


In [ ]:
time_axis = np.arange(BEAT_LENGTH) / TARGET_FS
plt.figure(figsize=(14, 5))
plt.plot(time_axis, sample_plot['clean'][:, 0], color='black', linewidth=1.8, label='Clean')
plt.plot(time_axis, sample_plot['noisy'][:, 0], color='gray', alpha=0.65, label='Noisy')
plt.plot(time_axis, sample_plot['denoised'][:, 0], color='crimson', linewidth=1.2, label='Denoised')
plt.title(f"Record {sample_plot['record']} - noise type 1 - {sample_plot['shot']}-shot")
plt.xlabel('Time (s)')
plt.ylabel('ECG amplitude')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plot_path = OUTPUT_DIR / 'qtdb_type1_trial_waveform.png'
plt.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()


## 13. Save trial results


In [ ]:
per_beat_path = OUTPUT_DIR / 'qtdb_type1_trial_metrics_per_beat.csv'
summary_numeric_path = OUTPUT_DIR / 'qtdb_type1_trial_summary_numeric.csv'
summary_display_path = OUTPUT_DIR / 'qtdb_type1_trial_summary_mean_std.csv'
metadata_path = OUTPUT_DIR / 'qtdb_type1_trial_metadata.csv'

metrics_per_beat.to_csv(per_beat_path, index=False)
summary_numeric.to_csv(summary_numeric_path, index=False)
summary_display.to_csv(summary_display_path, index=False)
metadata_frame.assign(noise_level=noise_levels, noise_start=noise_starts).to_csv(metadata_path, index=False)

run_info = {
    'checkpoint': str(CHECKPOINT_PATH),
    'checkpoint_epoch': checkpoint_epoch,
    'noise_type': 1,
    'shots': SHOTS,
    'beats': len(clean_beats),
    'max_beats_per_record': MAX_BEATS_PER_RECORD,
    'seed': SEED,
}
(OUTPUT_DIR / 'qtdb_type1_trial_run_info.json').write_text(
    json.dumps(run_info, indent=2), encoding='utf-8'
)
print('Saved results to:', OUTPUT_DIR)


## 14. Interpretation

This trial is useful for deciding whether to continue training type 1. The checkpoint should improve both cosine and SNR over `Noisy input`; a positive mean `SNR improvement` is required. `PRD_A_code` reproduces the denominator documented for Paper A, while `PRD_standard` uses clean-signal energy and is easier to compare with standard ECG literature.

A final Paper-A-style result still requires all test beats, both noise protocols, and the requested 1/3/5/10-shot aggregation.
